# 🎉 Public Events Data Preprocessing

Ingests raw public event data for Toronto, cleans and normalises it, and writes it through Bronze and Silver pipeline layers.

| Layer | Path | Description |
|---|---|---|
| **Bronze** | `data/bronze/public_events` | Raw CSV converted to Parquet |
| **Silver** | `data/silver/public_events` | Cleaned, normalised, feature-enriched Parquet |

| Section | Purpose |
|---|---|
| 1 | Import libraries |
| 2 | Load raw CSV and write Bronze |
| 3.1 | Inspect data info and unique values |
| 3.2 | Normalise column names, categories, and times |
| 3.3 | Remove duplicates |
| 3.4 | Filter to analysis date range and extract date features |
| 4 | Write Silver |


## Section 1 — Import Libraries

Loads `pandas` and `numpy` for data manipulation, `datetime` for date handling, and `pyarrow` for Parquet file writing.


In [0]:
import pandas as pd
import numpy as np
from datetime import datetime
import pyarrow as pa
import pyarrow.parquet as pq

## Section 2 — Load Raw Dataset & Write Bronze

Reads the raw public events CSV from the local data path and previews the first rows. The pandas DataFrame is then converted to a Spark DataFrame and written to the **Bronze** layer as Parquet — preserving the original data with no transformations applied.


In [0]:
df_raw = pd.read_csv("../../data/raw/public_events_toronto.csv")
df_raw.head()


In [0]:
spark_df = spark.createDataFrame(df_raw)
BRONZE_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/bronze/public_events"

spark_df.write.mode("overwrite").parquet(BRONZE_DIR)

## Section 3.1 — Data Inspection

Prints a summary of the DataFrame structure (column types, null counts, memory usage) and inspects unique values for `Event Category` and event time fields. This step confirms what normalisation is needed before cleaning.


In [0]:
df_raw.info()

In [0]:
df = df_raw.copy()
df['Event Category'].unique()

In [0]:
print("START TIMES", df['Event Time Starts'].unique())
print("\nEND TIMES:" , df['Event Time Ends'].unique())

## Section 3.2 — Data Normalisation

Three normalisation steps are applied:

1. **Column names** — stripped, lowercased, and spaces replaced with underscores (e.g. `"Event Category"` → `"event_category"`) for consistent programmatic access.
2. **Event categories** — multi-category values separated by `/` are simplified by keeping only the first category (e.g. `"Music/Arts"` → `"Music"`), reducing cardinality for downstream feature encoding.
3. **Event times** — `event_start_date` and `event_end_date` are parsed as proper datetime objects. Start and end times are rounded to the nearest hour and formatted as `HH:00` to align with the hourly granularity used by the bike share and weather datasets.


In [0]:
# Normaliztion of column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [0]:
# Normalization of event categories
df["event_category"] = df["event_category"].str.split("/").str[0].str.strip()
df['event_category'].unique()

In [0]:
# Transformation of timme
df['event_start_date'] = pd.to_datetime(df['event_start_date'], errors='coerce')
df['event_end_date'] = pd.to_datetime(df['event_end_date'], errors='coerce')

# Normalization of time
df["event_time_starts"] = pd.to_datetime(df["event_time_starts"], errors="coerce").dt.round("H").dt.strftime("%H:00")
df["event_time_ends"] = pd.to_datetime(df["event_time_ends"], errors="coerce").dt.round("H").dt.strftime("%H:00")



In [0]:
df.info()

## Section 3.3 — Data Cleaning

Removes exact duplicate rows from the dataset. Duplicates can arise from multiple data export runs or overlapping source files, and must be eliminated before writing to Silver to avoid inflating event counts in downstream joins.


In [0]:
df = df.drop_duplicates()

## Section 3.4 — Date Filtering & Feature Extraction

**Date filtering:** retains only events with a start date within the analysis window **October 2022 – September 2024**, matching the range used for model training and evaluation.

**Feature extraction:**
- A unique `event_id` is assigned from the DataFrame index and moved to the first column.
- `year`, `month`, and `day` columns are extracted from both `event_start_date` and `event_end_date`, enabling efficient time-based joins with the station-hour feature store without parsing full datetime objects at query time.


In [0]:
# Filtering by interested dates (OCT-2022 to SEP-2024)

start_range = pd.Timestamp("2022-10-01")
end_range = pd.Timestamp("2024-09-30")

df = df[
    (df["event_start_date"] >= start_range) &
    (df["event_start_date"] <= end_range)
]

In [0]:
df["event_id"] = df.index.astype(str)
df = df[["event_id"] + [col for col in df.columns if col != "event_id"]]

# Creation of new columns (YEAR, MONTH and DAY) based on event dates
df["event_start_date_y"] = df["event_start_date"].dt.year
df["event_start_date_m"] = df["event_start_date"].dt.month
df["event_start_date_d"] = df["event_start_date"].dt.day

df["event_end_date_y"] = df["event_end_date"].dt.year
df["event_end_date_m"] = df["event_end_date"].dt.month
df["event_end_date_d"] = df["event_end_date"].dt.day

In [0]:
df.head(10)

## Section 4 — Write Silver

The cleaned and feature-enriched DataFrame is saved in two formats:
1. **CSV** — written to `data/processed/public_events_cleaned.csv` for manual inspection or sharing.
2. **Parquet (Silver)** — converted to a Spark DataFrame and written to DBFS as the authoritative Silver layer for all downstream pipeline steps.


In [0]:
#save public events cleaned as csv in proccessed folder
df.to_csv("../../data/processed/public_events_cleaned.csv", index=False,encoding="utf-8")

In [0]:
spark_df_cleaned = spark.createDataFrame(df)
SILVER_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/public_events"

spark_df_cleaned.write.mode("overwrite").parquet(SILVER_DIR)